## Run a standard DCA model (fully connected) with StructureDCA
While StructureDCA is mainly build for **sparse DCA models inferred on protein 3D structure**, it can also be used as to run **standard DCA** models (fully-connected).

Indeed, this can be practical to run both standard and structure-informed DCA in the same framework.  
Moreover, we have observer that StructureDCA, when performing standard DCA, obtains results equal to other plmDCA frameworks like 'plmc' and 'pycofitness' while being around **3 times faster** to execute.

### Method 1: Hand-crafted standard DCA
An obvious, however slightly convoluted way to replicate standard DCA is just to set `distance_cutoff` to a very large number (to force all possible interactions to be contacts).  
However, this method still requires a protein 3D structure (while never using it).

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from structuredca import StructureDCA

In [ ]:
# Initialize fully-connected (standard DCA) with StructureDCA: handcrafted way
VERY_LARGE_NUMBER = 1000000000.0
sdca_full1 = StructureDCA(
    msa_path='../test_data/6acv_A_29-94.fasta',
    pdb_path='../test_data/6acv_A_29-94.pdb',
    distance_cutoff=VERY_LARGE_NUMBER,
    chains='A',
    num_threads=8,
    verbose=True,
)

### Method 2: Standard DCA without 3D structure
You can also ask StructureDCA to ignore protein 3D structure and to consider all possible interactions as contacts.  
Just set `pdb_path` as `None`.

In [ ]:
# Initialize fully-connected (standard DCA) with StructureDCA: structure-less way
sdca_full2 = StructureDCA(
    msa_path='../test_data/6acv_A_29-94.fasta',
    pdb_path=None,
    num_threads=8,
    verbose=True,
)

In [ ]:
# Let us verify that both methods lead to the same DCA model: here we predict dE for all single-site mutations
predictions1 = np.array([entry["StructureDCA"] for entry in sdca_full1.eval_mutations_table()])
predictions2 = np.array([entry["StructureDCA"] for entry in sdca_full2.eval_mutations_table()])

# NOTE: there could be some small deviations in predictions
# They are caused by small random fluctuation in the resolution of DCA coefficients h and J by Gradient Descent
rmse = np.sqrt(np.mean((predictions1 - predictions2)**2))

# Plot predictions of both models
print(rmse)
plt.title(f"RMSE={rmse:.4f}")
plt.scatter(predictions1, predictions2)
plt.show()


### Note: Usage of full vs. sparse J implementation
If you run standard (fully-connected) DCA, the usage of a sparse implementation for J is less relevent.  
You can change default parameter of StructureDCA: `use_sparse_J=False`.  
As consequence, the use of standard `numpy.NDArray` for couplings coefficients `J` will results in:
- Much more fast evaluations of mutations (however this step is rarely a computational-time problem)
- It will take `2` times more RAM than the SparseJ implementation (because the symmetry of J `J[i, j] = J[j, i].T` is not expoited).